# 👕 NeuroStartup — Classificador de roupas por foto

**Atividade Prática 3 · IA e Machine Learning · Prof. Hercules Ramos**

**Integrantes (nome e R.A.):**

| Nome | R.A. |
|---|---|
| (preencher) | (preencher) |
| (preencher) | (preencher) |
| (preencher) | (preencher) |
| (preencher) | (preencher) |

> Desafio escolhido: **1 · Classificar roupas por foto** (inspiração: Mercado Livre, Shein).

## O problema e o "cliente"
Um marketplace recebe milhares de fotos de vendedores por dia. Hoje, alguém precisa escolher a categoria de cada peça à mão, e categoria errada significa produto que ninguém encontra na busca.
**Cliente:** o time de catálogo do marketplace.
**Pergunta:** dá para sugerir a categoria da peça automaticamente a partir da foto?
**A rede prevê:** a categoria do produto (10 tipos), uma entre várias categorias, então a saída é `softmax` com 10 neurônios.

**Roteiro (os 5 passos da atividade):** 1 Conhecer os dados → 2 Baseline × rede → 3 Abrir o cofre → 4 Produto → 5 Documentação.

In [ ]:
# Imports e semente
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

SEED = 42
keras.utils.set_random_seed(SEED)
print("Keras", keras.__version__)

## 1 · Conhecer os dados
O dataset **Fashion-MNIST** já vem no Keras: fotos 28×28 em tons de cinza, com 10 categorias de roupas e acessórios.

In [ ]:
# Carregar os dados. O conjunto "teste" oficial (10 mil) vira o COFRE.
(X_resto, y_resto), (X_teste, y_teste) = keras.datasets.fashion_mnist.load_data()
CLASSES = ["Camiseta", "Calça", "Pulôver", "Vestido", "Casaco",
           "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"]
print("Resto:", X_resto.shape, "| Teste (cofre):", X_teste.shape)

In [ ]:
# Gráfico 1: um exemplo de cada categoria
fig, axs = plt.subplots(2, 5, figsize=(10, 4.6))
for i, ax in enumerate(axs.ravel()):
    ax.imshow(X_resto[np.where(y_resto == i)[0][0]], cmap="gray")
    ax.set_title(CLASSES[i]); ax.axis("off")
plt.suptitle("Um exemplo de cada categoria"); plt.tight_layout(); plt.show()

**Conclusão do gráfico 1:** as imagens são pequenas e centralizadas. Camiseta, camisa, pulôver e casaco são visualmente parecidas, então é onde esperamos mais confusão da rede.

In [ ]:
# Gráfico 2: quantas fotos de cada categoria
contagem = pd.Series(y_resto).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES)
contagem.plot(kind="bar", figsize=(9, 3), title="Fotos por categoria (base de treino)")
plt.ylabel("fotos"); plt.show()
print(contagem.to_string())

**Conclusão do gráfico 2:** as 10 categorias têm exatamente o mesmo número de fotos (6.000 cada). A base é balanceada, então a **acurácia** é uma métrica justa aqui (diferente de fraude, por exemplo).

### A regra do cofre 🔒
```
70.000 fotos
├── Teste (10.000) ─── 🔒 COFRE: só abre no final, uma única vez
└── Resto (60.000)
    ├── Treino  (48.000) ── a rede estuda aqui
    └── Validação (12.000) ── o "simulado" para escolher a melhor rede
```

In [ ]:
# Separar treino / validação (o teste já está no cofre) e escalar pixels de 0-255 para 0-1
X_treino, X_valid, y_treino, y_valid = train_test_split(
    X_resto, y_resto, test_size=0.2, random_state=SEED, stratify=y_resto)

X_treino = X_treino.astype("float32") / 255.0
X_valid  = X_valid.astype("float32") / 255.0
X_teste  = X_teste.astype("float32") / 255.0      # 🔒 só será usado no passo 3

print("Treino:", X_treino.shape, "| Validação:", X_valid.shape, "| Teste (cofre):", X_teste.shape)

## 2 · Baseline × rede
Antes da rede, medimos modelos simples. A rede só vale a pena se bater o baseline.
Os modelos simples treinam com 10 mil fotos (achatadas em 784 números) para rodar rápido.

In [ ]:
def achatar(X):
    return X.reshape(len(X), -1)

N = 10000
dummy     = DummyClassifier(strategy="most_frequent").fit(achatar(X_treino[:N]), y_treino[:N])
logistica = LogisticRegression(max_iter=200).fit(achatar(X_treino[:N]), y_treino[:N])   # baseline principal
arvore    = DecisionTreeClassifier(max_depth=12, random_state=SEED).fit(achatar(X_treino[:N]), y_treino[:N])  # bônus

registros = []
for nome, m in [("Chute (classe mais comum)", dummy), ("Regressão logística (baseline)", logistica), ("Árvore de decisão", arvore)]:
    acc = m.score(achatar(X_valid), y_valid)
    registros.append({"tipo": "baseline", "modelo": nome, "parametros": None, "epocas_treinadas": None,
                      "perda_validacao": None, "acuracia_validacao": round(acc, 4)})
    print(f"{nome:32s} acurácia na validação: {acc:.1%}")

Agora as redes em Keras: **4 arquiteturas** diferentes, com dropout (jogadores faltando) e early stopping (parar no melhor simulado). Cada tentativa vira uma linha no `experimentos.csv`.

In [ ]:
def criar_rede(camadas, dropout=0.2):
    m = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten()])
    for n in camadas:
        m.add(layers.Dense(n, activation="relu"))
        m.add(layers.Dropout(dropout))
    m.add(layers.Dense(10, activation="softmax"))      # 10 categorias -> softmax
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

CONFIGS = [(64,), (128,), (256, 128), (512, 256)]
redes, historicos = {}, {}

for camadas in CONFIGS:
    keras.utils.set_random_seed(SEED)
    rede = criar_rede(camadas)
    parada = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    hist = rede.fit(X_treino, y_treino, validation_data=(X_valid, y_valid),
                    epochs=50, batch_size=128, verbose=0, callbacks=[parada])
    perda_val, acc_val = rede.evaluate(X_valid, y_valid, verbose=0)
    nome = f"Rede {camadas}"
    redes[nome], historicos[nome] = rede, hist
    registros.append({"tipo": "rede", "modelo": nome, "parametros": rede.count_params(),
                      "epocas_treinadas": len(hist.history["loss"]),
                      "perda_validacao": round(perda_val, 4), "acuracia_validacao": round(acc_val, 4)})
    print(f"{nome:18s} épocas: {len(hist.history['loss']):2d} | acurácia na validação: {acc_val:.1%}")

experimentos = pd.DataFrame(registros)
experimentos.to_csv("experimentos.csv", index=False)
experimentos

In [ ]:
# Curvas de aprendizado de todas as redes (treino × validação)
fig, axs = plt.subplots(2, 2, figsize=(11, 6), sharey=True)
for ax, (nome, h) in zip(axs.ravel(), historicos.items()):
    ax.plot(h.history["loss"], label="treino"); ax.plot(h.history["val_loss"], label="validação")
    ax.set_title(nome); ax.set_xlabel("época"); ax.set_ylabel("perda"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Como ler:** treino e validação descendo juntos = a rede está aprendendo. Se a validação sobe enquanto o treino desce, a rede começou a decorar, e o early stopping interrompe o treino nesse ponto e devolve os pesos da melhor época.
(Conferir nas curvas acima e anotar aqui a conclusão do grupo: qual rede começou a decorar mais cedo? a maior foi a melhor?)

## 3 · Abrir o cofre 🔓 (uma vez só!)
Escolhemos a melhor rede **pela validação** e só então olhamos o teste, uma única vez, para todos os modelos ao mesmo tempo.

In [ ]:
# Abrir o cofre do teste (UMA vez)
tabela_redes = experimentos[experimentos["tipo"] == "rede"].sort_values("acuracia_validacao", ascending=False)
melhor = tabela_redes.iloc[0]["modelo"]
modelo_final = redes[melhor]

acc_rede  = modelo_final.evaluate(X_teste, y_teste, verbose=0)[1]
acc_log   = logistica.score(achatar(X_teste), y_teste)
acc_arvore = arvore.score(achatar(X_teste), y_teste)
acc_chute = dummy.score(achatar(X_teste), y_teste)

print(f"Melhor arquitetura (escolhida na validação): {melhor}")
print(f"TESTE -> chute: {acc_chute:.1%} | árvore: {acc_arvore:.1%} | regressão logística (baseline): {acc_log:.1%} | REDE: {acc_rede:.1%}")

erro_rede, erro_log = 1 - acc_rede, 1 - acc_log
reducao = 1 - erro_rede / erro_log

FRASE_NEGOCIO = (f"A cada 100 peças fotografadas, a rede sugere a categoria certa em {acc_rede*100:.0f}, "
                 f"contra {acc_log*100:.0f} do modelo simples: são {(erro_log-erro_rede)*100:.0f} erros a menos "
                 f"a cada 100 peças, ou {reducao:.0%} menos revisão manual no catálogo.")
print("\n💼", FRASE_NEGOCIO)

In [ ]:
# Onde a rede erra? (matriz de confusão no teste)
from sklearn.metrics import confusion_matrix
pred = modelo_final.predict(X_teste, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_teste, pred)
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=90); plt.yticks(range(10), CLASSES)
plt.xlabel("previsto"); plt.ylabel("real"); plt.colorbar(); plt.title("Matriz de confusão (teste)"); plt.tight_layout(); plt.show()

acc_classe = pd.Series(cm.diagonal() / cm.sum(axis=1), index=CLASSES).sort_values()
print("Acurácia por categoria (pior -> melhor):"); print(acc_classe.map("{:.1%}".format).to_string())

## 4 · Produto
Um modelo treinado vira um **arquivo**, e o produto é *modelo salvo + função de previsão + interface*.

In [ ]:
# Salvar e carregar o modelo
modelo_final.save("modelo_roupas.keras")
carregado = keras.saving.load_model("modelo_roupas.keras")
print("Arquivo salvo:", os.path.getsize("modelo_roupas.keras") // 1024, "KB")
print("Mesmas previsões?", np.allclose(carregado.predict(X_teste[:5], verbose=0),
                                        modelo_final.predict(X_teste[:5], verbose=0), atol=1e-5))

In [ ]:
# Função de previsão + painel interativo
from ipywidgets import interact, IntSlider

def classificar_roupa(imagem):
    """Recebe uma imagem 28x28 em tons de cinza e devolve as 3 categorias mais prováveis."""
    x = np.asarray(imagem, dtype="float32").reshape(1, 28, 28)
    if x.max() > 1: x = x / 255.0
    p = carregado.predict(x, verbose=0)[0]
    top = np.argsort(p)[::-1][:3]
    return [(CLASSES[i], float(p[i])) for i in top]

def painel(indice=0):
    img = X_teste[indice]
    resultado = classificar_roupa(img)
    plt.figure(figsize=(2.6, 2.6)); plt.imshow(img, cmap="gray"); plt.axis("off")
    plt.title(f"Categoria real: {CLASSES[y_teste[indice]]}"); plt.show()
    print(f"🛍️ Palpite: {resultado[0][0]} ({resultado[0][1]:.0%} de confiança)")
    for nome, prob in resultado[1:]:
        print(f"   também pode ser: {nome} ({prob:.0%})")

painel(0)
interact(painel, indice=IntSlider(min=0, max=len(X_teste) - 1, step=1, value=0, description="foto nº"));

📸 **Tirar um print desta tela com o painel funcionando** e salvar no repositório como `print_tela.png` (ele aparece no README).

## 5 · Documentação — Model card (a "bula" do modelo)
O arquivo `MODEL_CARD.md` é gerado a partir dos resultados reais deste notebook.

In [ ]:
card = f'''# Model Card — Classificador de Categoria de Roupas

- **Uso pretendido:** sugerir a categoria de uma peça (10 tipos) a partir de uma foto, para o time de catálogo revisar mais rápido. A sugestão é de apoio: uma pessoa confirma.
- **Não usar para:** punir, suspender ou ranquear vendedores por "categoria errada"; identificar ou classificar pessoas; decidir sozinho, sem revisão humana, onde um produto aparece na loja; classificar fotos reais de celular sem re-treinar (veja Limitações).
- **Dados:** Fashion-MNIST (Zalando), 70.000 fotos 28x28 em tons de cinza, 10 categorias balanceadas. Divisão: 48.000 treino, 12.000 validação, 10.000 teste (cofre, usado uma única vez).
- **Modelo:** rede neural em Keras {melhor} com dropout 0.2 e early stopping; comparada a regressão logística (baseline) e árvore de decisão.
- **Como medir se está bom:** acurácia no teste e acurácia por categoria (matriz de confusão). Teste: rede {acc_rede:.1%}, regressão logística {acc_log:.1%}, árvore {acc_arvore:.1%}, chute {acc_chute:.1%}. Categoria mais difícil: {acc_classe.index[0]} ({acc_classe.iloc[0]:.1%}).
- **Resultado em linguagem de negócio:** {FRASE_NEGOCIO}
- **Limitações:** treinada só com fotos pequenas, centralizadas, de peça única e em tons de cinza; fotos reais (fundo, luz, vários itens) vão piorar o resultado. Confunde peças parecidas (camiseta, camisa, pulôver, casaco). Só conhece 10 categorias. Confiança alta não é certeza. Re-treinar com fotos reais do catálogo e monitorar o erro depois de publicar.
'''
open("MODEL_CARD.md", "w", encoding="utf-8").write(card)
print(card)

## ✅ Checklist antes de entregar
- [ ] Nome e R.A. de todos no README (e na tabela do topo deste notebook)
- [ ] Notebook **rodado do início ao fim** (Kernel → Restart & Run All), com os resultados aparecendo
- [ ] `experimentos.csv` com pelo menos 3 linhas
- [ ] Frase de negócio com o resultado do teste
- [ ] Print da tela funcionando no README
- [ ] `MODEL_CARD.md` com o "não usar para"
- [ ] Declaração de uso de IA no README
- [ ] Nenhuma senha ou chave no código
- [ ] Teste final: alguém de fora do grupo consegue seguir o README